# Nutosis

It is October 2018. The squirrels in Central Park are running into a problem and we need your help.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help).

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes the train or test set and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preparation should be noted here but actually implemented in the Preparation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit the train/test parquet files and a pickled preprocessor and final model along with these notebooks.

# Frame the Problem and Look at the Big Picture

1. Define the objective in business terms.
The wildlife conservation team needs a way to catch sick squirrels early even though they are short staffed. Our goal is to use volunteer sighting reports to flag squirrels that are likely sick, so staff can focus their limited time on the squirrels most worth capturing and testing instead of checking at random. Success means catching more sick squirrels early with the same number of staff hours and wasting fewer captures on healthy ones.

2. How will your solution be used?
Volunteers submit hundreds of sighting reports each day, and staff can capture and test about 40–50 squirrels per day. The model would score each report as it arrives so staff can decide whether to send a team. Staff could review borderline cases based on the squirrel’s location, the sighting time, and the day’s remaining capacity. As new test results arrive, the team could use them to evaluate and update the model.

3. What are the current solutions/workarounds (if any)?
The wildlife conservation team uses reported sightings and decide if its a credible source. If the team deems it as credible they go out, test the squirrels immune system and euthanize the squirrel at their lab if they show signs of Nutosis. 

4. How should you frame this problem (supervised/unsupervised, online/offline, …)?
This is a supervised learning problem because we have labeled outcomes from captures and testing that tell us whether a squirrel was sick. We would start with binary classification (sick or healthy) and could move to a multi class approach since Nutosis is evolving so (healthy, early stage, and advanced). Since new reports keep coming in, it fits a online batch setup where the model scores and retrains every few hours. 

5. How should performance be measured? Is the performance measure aligned with the business objective?
Our main metric will be recall because the client wants to miss as few sick sightings as possible. We will measure recall within the team's daily limit of 40-50 captures and use precision to see what percentage of the selected sightings are labeled as sick, since each trip uses limited staff time. We will also report F1 when comparing models.

6. What would be the minimum performance needed to reach the business objective?
The client did not specify a minimum recall target. We would consider the model useful if it identifies more sick sightings than the team's current behavior-based approach while staying within the same daily limit of 40-50 captures.

7. What are comparable problems? Can you reuse experience or tools?
This problem is similar to prioritizing emergency calls because the team has to decide which reports need attention first when staff are limited. We can use pandas to organize the sighting data, NumPy for calculations, Matplotlib to plot patterns in the reports, and scikit-learn to train and compare models.

8. Is human expertise available?
Since Nutosis is a new virus, the only way to diagnose a squirrel is by testing its immune system. The staff do not have the expertise to spot signs of Nutosis from a squirrel's appearance.

9. How would you solve the problem manually?
Staff would go through each volunteer report as it comes in, remove duplicate sightings of the same squirrel, and look for patterns like clusters of reports in the same area or unusual behavior. Based on that, they would pick which squirrels to capture, test their immune systems, and record whether each one was sick. Over time they would keep track of where positive cases showed up and use that to decide where to look next. 

10. List the assumptions you (or others) have made so far. Verify assumptions if possible.
We assume that volunteer reports contain enough detail to help us identify sick squirrels. We will check this by comparing the reported behaviors with the disease labels. The client confirmed that each row represents one sighting and that the same squirrel can be reported more than once. For this project, we treat sightings listed in diseased.csv as sick and all other sightings as healthy. All 379 IDs in the disease file matched the census data, giving us 379 sightings labeled as sick and 2,644 labeled as healthy.

# Get the Data

1. List the data you need and how much you need
    1. Squirrel census: We need all 3,023 Central Park sightings recorded from October 6–20, 2018.
    2. Disease labels: We need all 379 sighting IDs in diseased.csv. We will label matching sightings as sick and all other sightings as healthy.
    3. Weather: We need Central Park temperature and precipitation records for the census period. We will use only weather information available before each sighting.
    4. Park events: We need Central Park event permit dates and times to count how many events are scheduled during each census shift.
    5. Place locations: We need coordinates for the Central Park places named in the event permits to measure how close each event was to each sighting.

2. Find and document where you can get that data
    1. Squirrel census: We will use the 2018 Central Park squirrel sightings published by [NYC Open Data](https://data.cityofnewyork.us/Environment/2018-Central-Park-Squirrel-Census-Squirrel-Data/vfnx-vebw/about_data).
    2. Disease labels: Our client provided diseased.csv, which contains the sighting IDs labeled as sick.
    3. Weather: We will get temperature and precipitation records from NOAA’s [Central Park weather station](https://www.ncdc.noaa.gov/cdo-web/datasets/GHCND/stations/GHCND%3AUSW00094728/detail).
    4. Park events: We will get event locations and times from NYC Open Data’s [historical event permits](https://data.cityofnewyork.us/City-Government/NYC-Permitted-Event-Information-Historical/bkfu-528j/data).
    5. Place locations: We will get coordinates for the permit locations from [OpenStreetMap](https://www.openstreetmap.org/way/427818536) through the Overpass API.

3. Get access authorizations
The squirrel census, weather records, and event permits are publicly available through NYC Open Data and NOAA. OpenStreetMap data is free to use under the [Open Database License](https://www.openstreetmap.org/copyright). Our client provided diseased.csv for use in this project.

4. Create a workspace (with enough storage space) <br>
We will use our local project folder to run the notebooks and cache downloaded data for reuse.

5. Get the data
We load all four files from our project folder and download them from our Kaggle dataset if they are missing. We download the place locations from OpenStreetMap the first time and save them as central_park_osm_places.csv. Our data covers these dates:
    1. Squirrel census: sightings from October 6–20, 2018.
    2. Disease labels: sick sighting IDs from the same October 6–20, 2018 census.
    3. Weather: daily records from October 5–19, 2018, one day before each census date.
    4. Park events: Central Park permits that overlap October 6–20, 2018.

6. Convert the data to a format you can easily manipulate (without changing the data itself)
We load the squirrel census, disease IDs, weather records, and event permits into pandas DataFrames. We read diseased.csv without a header so the first sighting ID is kept as data.

7. Ensure sensitive information is deleted or protected (e.g. anonymized)
We use sighting IDs to match reports with disease labels. We check the free-text notes for names or contact details and remove any we find before saving the training and test files. We keep the original data files out of the submission repository.

In [25]:
%pip install kagglehub scikit-learn matplotlib numpy ipywidgets pandas pyarrow requests
import os
import shutil
import kagglehub
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Functions

In [26]:
URL = "jamellalvarez/project2nutosisdataset/versions/3"

def list_dataset_files():
    """Prints the files in the Kaggle dataset so you know which names to use."""
    cache_dir = kagglehub.dataset_download(URL)
    files = sorted(os.listdir(cache_dir))
    for f in files:
        print(f)
    return files

def get_local_file(filename):
    """
    Checks if filename exists locally. If not, fetches it from
    the Kaggle cache and copies it to your project directory.
    """
    if not os.path.exists(filename):
        print(f"Fetching {filename} from Kaggle...")
        cache_dir = kagglehub.dataset_download(URL)
        cached_file_path = os.path.join(cache_dir, filename)
        shutil.copy(cached_file_path, filename)
    return filename

def read_squirrel_data():
    """Loads the squirrel census sightings."""
    return pd.read_csv(get_local_file("2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261104.csv"))

def read_sickness_data():
    """Loads the Nutosis test results."""
    return pd.read_csv(get_local_file("diseased.csv"), header=None, names=["sighting_id"])

def read_weather_data():
    """Loads the Central Park daily weather for October 5-19, 2018."""
    return pd.read_csv(
        get_local_file("central_park_weather_2018-10-05_to_2018-10-19.csv"),
        usecols=["DATE", "TMIN", "TMAX", "PRCP"],
        parse_dates=["DATE"],
    )

def read_events_data():
    """Loads the Central Park event permits that overlap October 6-20, 2018."""
    return pd.read_csv(
        get_local_file("central_park_events_2018-10-06_to_2018-10-20.csv"),
        parse_dates=["start_date_time", "end_date_time"],
    )

def read_place_coords():
    """Load the cached OpenStreetMap coordinates from our Kaggle dataset."""
    return pd.read_csv(get_local_file("central_park_osm_places.csv"))

# Data 

In [27]:
squirrel_data = read_squirrel_data()
sickness_data = read_sickness_data()

print("Squirrel data:", squirrel_data.shape)
print("Sickness data:", sickness_data.shape)
squirrel_data.head()

Squirrel data: (3023, 31)
Sickness data: (379, 1)


,X,Y,Unique Squirrel ID,Hectare,Shift,Date,Hectare Squirrel Number,Age,Primary Fur Color,Highlight Fur Color,...,Kuks,Quaas,Moans,Tail flags,Tail twitches,Approaches,Indifferent,Runs from,Other Interactions,Lat/Long
0,-73.956134,40.794082,37F-PM-1014-03,37F,PM,10142018,3,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9561344937861 40.7940823884086)
1,-73.968857,40.783783,21B-AM-1019-04,21B,AM,10192018,4,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9688574691102 40.7837825208444)
2,-73.974281,40.775534,11B-PM-1014-08,11B,PM,10142018,8,NaN,Gray,NaN,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.97428114848522 40.775533619083)
3,-73.959641,40.790313,32E-PM-1017-14,32E,PM,10172018,14,Adult,Gray,NaN,...,False,False,False,False,False,False,False,True,NaN,POINT (-73.9596413903948 40.7903128889029)
4,-73.970268,40.776213,13E-AM-1017-05,13E,AM,10172018,5,Adult,Gray,Cinnamon,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9702676472613 40.7762126854894)


In [28]:
squirrel_data.info()
sickness_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3023 entries, 0 to 3022
Data columns (total 31 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   X                                           3023 non-null   float64
 1   Y                                           3023 non-null   float64
 2   Unique Squirrel ID                          3023 non-null   str    
 3   Hectare                                     3023 non-null   str    
 4   Shift                                       3023 non-null   str    
 5   Date                                        3023 non-null   int64  
 6   Hectare Squirrel Number                     3023 non-null   int64  
 7   Age                                         2902 non-null   str    
 8   Primary Fur Color                           2968 non-null   str    
 9   Highlight Fur Color                         1937 non-null   str    
 10  Combination of Primary 

### Central Park weather

The client noticed fewer reports on colder days. We collect daily minimum temperature, maximum temperature, and precipitation from the Central Park weather station. We use the previous day’s weather because the full daily summary would not be available when a morning sighting is reported. For sightings from October 6–20, we need weather from October 5–19.

The data comes from NOAA's daily summaries for the Central Park weather station.

In [29]:
weather_data = read_weather_data()
print(f"{len(weather_data)} weather records")
weather_data.head(3)

15 weather records


,DATE,PRCP,TMAX,TMIN
0,2018-10-05,0.0,21.7,15.6
1,2018-10-06,0.0,20.0,17.2
2,2018-10-07,0.0,26.1,19.4


### Central Park events

The client told us that sightings depend on where and when people visit the park. We collect permits for Central Park events that overlap the census dates, including events that started before October 6. A permit tells us an event was scheduled. Attendance is unknown.

The data comes from NYC Open Data's historical event permits.

In [30]:
events_data = read_events_data()
print(f"{len(events_data)} event records")
events_data[["event_name", "event_location", "start_date_time"]].head(3)

521 event records


,event_name,event_location,start_date_time
0,Central Park Squirrel Census,Central Park: Mineral Springs,2018-10-06 07:00:00
1,Grete's Great Gallop 10K,"Central Park: 72nd Street Cross Drive ,Central...",2018-10-06 08:00:00
2,Soccer - Non Regulation,"Central Park: North Meadow-Soccer-05 ,Central ...",2018-10-06 08:00:00


In [31]:
project_data = squirrel_data.copy()
disease_ids = sickness_data["sighting_id"]

project_data["diseased"] = project_data["Unique Squirrel ID"].isin(disease_ids)
matched_ids = disease_ids.isin(project_data["Unique Squirrel ID"]).sum()

print(f"{matched_ids} of {len(disease_ids)} disease IDs matched.")
print(project_data["diseased"].value_counts())

379 of 379 disease IDs matched.
diseased
False    2644
True      379
Name: count, dtype: int64


### Match sightings with the previous day’s weather

We match each sighting to the previous day’s weather because the full weather summary for the sighting day would not yet be available when the report came in.

In [32]:
# Date is stored as an integer like 10142018 (MMDDYYYY), so we convert it to a real date.
project_data["Date"] = pd.to_datetime(project_data["Date"].astype(str), format="%m%d%Y")
sighting_dates = project_data["Date"]
previous_dates = sighting_dates - pd.Timedelta(days=1)
weather_by_date = weather_data.set_index("DATE")

project_data["prev_tmin_c"] = previous_dates.map(weather_by_date["TMIN"])
project_data["prev_tmax_c"] = previous_dates.map(weather_by_date["TMAX"])
project_data["prev_precip_mm"] = previous_dates.map(weather_by_date["PRCP"])

project_data[["Date", "prev_tmin_c", "prev_tmax_c", "prev_precip_mm"]].head(3)

,Date,prev_tmin_c,prev_tmax_c,prev_precip_mm
0,2018-10-14,7.8,12.8,2.5
1,2018-10-19,5.6,10.0,0.0
2,2018-10-14,7.8,12.8,2.5


### Match sightings with scheduled park events

Every census shift overlaps at least one Central Park event permit, so we count distinct permits for each shift. We treat AM as midnight to noon and PM as noon to midnight. Each sighting receives the count for its date and shift. This count describes scheduled activity across the park.

In [33]:
shift_hours = project_data["Shift"].map({"AM": 0, "PM": 12})
shift_starts = sighting_dates + pd.to_timedelta(shift_hours, unit="h")

event_counts = {}
for start in shift_starts.unique():
    end = start + pd.Timedelta(hours=12)
    overlaps = (
        (events_data["start_date_time"] < end)
        & (events_data["end_date_time"] > start)
    )
    event_counts[start] = events_data.loc[overlaps, "event_id"].nunique()

project_data["park_event_count"] = shift_starts.map(event_counts)
project_data[["Date", "Shift", "park_event_count"]].drop_duplicates().head()

,Date,Shift,park_event_count
0,2018-10-14,PM,27
1,2018-10-19,AM,10
3,2018-10-17,PM,32
4,2018-10-17,AM,11
5,2018-10-10,AM,10


### Match sightings with nearby park events

The park wide count above gives every sighting in a shift the same number, so it says nothing about where events happened. We look up each permit location in OpenStreetMap and compare it with each sighting's latitude and longitude. nearby_event_count: distinct events within 500 m that overlap the sighting's shift. nearest_event_meters: distance in meters to the closest event active during the shift.All coordinates come from OpenStreetMap, downloaded with an Api.The code only matches each permit place name to an OpenStreetMap name. Most places have the same name in OpenStreetMap, and numbered fields like North Meadow-Soccer-01 use the OpenStreetMap field with the same number. Lawns, picnic areas and parts of a place use the feature they are named after, so for example Turtle Pond Lawn uses the center of Turtle Pond. Tennis permits only give a court number, so they use the Tennis Center, which has 30 of the 32 tennis courts OpenStreetMap shows in the park. Locations outside Central Park, long roads and sidewalks with no single point like West Drive, and names OpenStreetMap doesn't have like The Mall are skipped. 10 of 230 permits (about 4%) are only on roads or sidewalks, or at places OpenStreetMap doesn't name, so nearby_event_count leaves them out.park_event_count still counts them.

In [34]:
# Location of each permit place, from OpenStreetMap (see read_place_coords).
osm_places = read_place_coords()

# Places with the same name in OpenStreetMap.
same_name = [
    "Bow Bridge", "Cedar Hill", "Columbus Circle", "Cop Cot", "East Meadow",
    "Frederick Douglass Circle", "Gapstow Bridge", "Great Hill Oval", "Kerbs Boathouse",
    "Ladies' Pavilion", "Mineral Springs", "Rumsey Playfield", "Shakespeare Garden",
    "Summit Rock", "Tavern on the Green", "Wagner Cove",
]
permit_to_osm = {
    **{place: place for place in same_name},
    "Arsenal": "The Arsenal",
    "Hernshead": "The Hernshead",
    # Numbered fields use the OpenStreetMap field with the same number.
    **{f"North Meadow-Soccer-0{n}": f"North Meadow Soccer Field {n}" for n in range(1, 7)},
    **{f"Heckscher Fields-Softball-0{n}": f"Heckscher Ballfield {n}" for n in range(1, 7)},
    **{f"Great Lawn-Softball-0{n}": f"Great Lawn Ball Field {n}" for n in range(1, 6)},
    # Lawns, picnic areas and parts of a place use the feature they are named after.
    "Adventure Playground Lawn": "Adventure Playground",
    "Bandshell Plaza": "Naumburg Bandshell",
    "Bethesda Fountain Terrace": "Bethesda Terrace",
    "Cherry Hill": "Cherry Hill Fountain",
    "Cleopatra's Needle Lawn": "Cleopatra's Needle",
    "Dene Lawn": "The Dene",
    "East 102nd St Clay Oval": "East 102nd Street",
    "East 72nd Street Playground Lawns": "East 72nd Street Playground",
    "East Meadow Clay Area": "East Meadow",
    "East Meadow Picnic Zone": "East Meadow",
    "Great Hill North": "Great Hill",
    "Great Lawn Plgd-Basketball-05": "The Great Lawn",
    "Great Lawn South West Lawn": "The Great Lawn",
    "Harlem Meer Picnic Area": "Harlem Meer",
    "Heckscher Playground Lawns": "Hecksher Playground",  # spelled this way in OpenStreetMap
    "Johnson Playground Lawns": "Billy Johnson Playground",
    "King Jagiallo Plaza": "King Jagiello Monument",
    "Lawn North of Levin Playground": "James Michael Levin Playground",
    "Ross Pinetum East": "Arthur Ross Pinetum",
    "Ross Pinetum West": "Arthur Ross Pinetum",
    "Rudin Playground Lawn": "Rudin Family Playground",
    "The Pond Lawns": "The Pond",
    "The Pool Lawns": "The Pool",
    "Turtle Pond Lawn": "Turtle Pond",
    "West 96th Street Entrance": "West 96th Street",
    "West 96th Street Landscape": "West 96th Street",
    "Wild West Playground Lawn": "Tarr-Coyne Wild West Playground",
    # Tennis permits only give a court number (5 to 8). OpenStreetMap has 30 tennis courts
    # at the Tennis Center and only 2 elsewhere in the park, so we use the Tennis Center.
    **{f"Tennis-0{n}": "Central Park Tennis Center" for n in range(5, 9)},
}
# Roads, sidewalks, and names OpenStreetMap doesn't have (The Mall, Merchants' Gate, Bowling Green Lawn, Heckscher Picnic Area, North Reservoir Landscape A and C) are left out.
# Some places are mapped as several pieces, so we average their center points.
osm_points = osm_places.groupby("name")[["lat", "lon"]].mean()
missing = sorted(set(permit_to_osm.values()) - set(osm_points.index))
print("OpenStreetMap names not found:", missing)
place_coords = {
    place: tuple(osm_points.loc[osm_name])
    for place, osm_name in permit_to_osm.items()
    if osm_name in osm_points.index
}

# One row per (event, Central Park location). Permits can list several places,
# including fields in other parks, which we drop. The census itself is not counted.
event_places = events_data[events_data["event_name"] != "Central Park Squirrel Census"].copy()
event_places["place"] = event_places["event_location"].str.split(",")
event_places = event_places.explode("place")
event_places["place"] = event_places["place"].str.strip()
event_places = event_places[event_places["place"].str.startswith("Central Park:")]
event_places["place"] = event_places["place"].str.removeprefix("Central Park:").str.strip()
event_places["lat"] = event_places["place"].map(lambda p: place_coords.get(p, (np.nan, np.nan))[0])
event_places["lon"] = event_places["place"].map(lambda p: place_coords.get(p, (np.nan, np.nan))[1])
unmapped = event_places.loc[event_places["lat"].isna(), "place"].unique()
print("Places without coordinates (skipped):", sorted(unmapped))
event_places = event_places.dropna(subset=["lat", "lon"])

def distance_m(lat1, lon1, lat2, lon2):
    """Haversine distance in meters (works on NumPy arrays)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6_371_000 * 2 * np.arcsin(np.sqrt(a))

NEARBY_RADIUS_M = 500
project_data["nearby_event_count"] = 0
project_data["nearest_event_meters"] = np.nan

for start, rows in project_data.groupby(shift_starts).groups.items():
    end = start + pd.Timedelta(hours=12)
    active = event_places[(event_places["start_date_time"] < end) & (event_places["end_date_time"] > start)]
    if active.empty:
        continue
    dist = distance_m(
        project_data.loc[rows, "Y"].to_numpy()[:, None],
        project_data.loc[rows, "X"].to_numpy()[:, None],
        active["lat"].to_numpy()[None, :],
        active["lon"].to_numpy()[None, :],
    )
    # Count each event once, even if several of its locations are nearby.
    near = pd.DataFrame(dist <= NEARBY_RADIUS_M).T.groupby(active["event_id"].to_numpy()).any()
    project_data.loc[rows, "nearby_event_count"] = near.sum(axis=0).to_numpy()
    project_data.loc[rows, "nearest_event_meters"] = dist.min(axis=1).round()

project_data[["Unique Squirrel ID", "Shift", "nearby_event_count", "nearest_event_meters"]].head()

Fetching central_park_osm_places.csv from Kaggle...


100%|██████████| 178k/178k [00:00<00:00, 3.18MB/s]

Extracting files...
OpenStreetMap names not found: []
Places without coordinates (skipped): ['102nd Street Transverse', '72nd Street Cross Drive', 'Bowling Green Lawn', 'Center Drive', 'Center Loop', 'Central Park West Sidewalks - 97th St to 110th St', 'Dead Road North', 'East Drive', 'Fifth Ave Sidewalks - 60 St to 65 St', 'Fifth Ave Sidewalks - 65th St to 72nd St', 'Heckscher Picnic Area', 'Lower Loop', "Merchants' Gate", 'North Reservoir Landscape A', 'North Reservoir Landscape C', 'South Dead Road A', 'The Mall', 'West Drive']


,Unique Squirrel ID,Shift,nearby_event_count,nearest_event_meters
0,37F-PM-1014-03,PM,11,136.0
1,21B-AM-1019-04,AM,0,890.0
2,11B-PM-1014-08,PM,6,81.0
3,32E-PM-1017-14,PM,20,209.0
4,13E-AM-1017-05,AM,4,136.0


In [35]:
added_columns = ["prev_tmin_c", "prev_tmax_c", "prev_precip_mm", "park_event_count", "nearby_event_count", "nearest_event_meters"]

print("Total sightings:", len(project_data))
print("Sick sightings:", project_data["diseased"].sum())
print("Missing values per added column:")
print(project_data[added_columns].isna().sum())

Total sightings: 3023
Sick sightings: 379
Missing values per added column:
prev_tmin_c             0
prev_tmax_c             0
prev_precip_mm          0
park_event_count        0
nearby_event_count      0
nearest_event_meters    0
dtype: int64


In [36]:
project_data.head(5)

,X,Y,Unique Squirrel ID,Hectare,Shift,Date,Hectare Squirrel Number,Age,Primary Fur Color,Highlight Fur Color,...,Runs from,Other Interactions,Lat/Long,diseased,prev_tmin_c,prev_tmax_c,prev_precip_mm,park_event_count,nearby_event_count,nearest_event_meters
0,-73.956134,40.794082,37F-PM-1014-03,37F,PM,2018-10-14,3,NaN,NaN,NaN,...,False,NaN,POINT (-73.9561344937861 40.7940823884086),False,7.8,12.8,2.5,27,11,136.0
1,-73.968857,40.783783,21B-AM-1019-04,21B,AM,2018-10-19,4,NaN,NaN,NaN,...,False,NaN,POINT (-73.9688574691102 40.7837825208444),False,5.6,10.0,0.0,10,0,890.0
2,-73.974281,40.775534,11B-PM-1014-08,11B,PM,2018-10-14,8,NaN,Gray,NaN,...,False,NaN,POINT (-73.97428114848522 40.775533619083),False,7.8,12.8,2.5,27,6,81.0
3,-73.959641,40.790313,32E-PM-1017-14,32E,PM,2018-10-17,14,Adult,Gray,NaN,...,True,NaN,POINT (-73.9596413903948 40.7903128889029),False,8.3,15.6,0.3,32,20,209.0
4,-73.970268,40.776213,13E-AM-1017-05,13E,AM,2018-10-17,5,Adult,Gray,Cinnamon,...,False,NaN,POINT (-73.9702676472613 40.7762126854894),False,8.3,15.6,0.3,11,4,136.0


# Train and Test Split

In [37]:
train_set, test_set = train_test_split(
    project_data, test_size=0.2, stratify=project_data["diseased"], random_state=42
)
train_set.to_parquet("train_set.parquet", index=False)
test_set.to_parquet("test_set.parquet", index=False)

We used an 80/20 split with a stratify on diseased. Only about 12.5% of the sightings are sick, so we stratified on the diseased column to make sure both sets have the same share of sick squirrels. Fur color is really skewed since most black and cinnamon squirrels are sick while very few gray ones are, but the split keeps each fur color within a few percentage points between the two sets. The other columns like date, shift, age, and behaviors were also close enough, so stratifying on the label was enough.